# BIAS IN BIOS — Dataset Exploration
Dataset: 396,190 rows with columns: split, profession, profession_name, gender, gender_name, hard_text

## CELL 1: Imports & Setup

In [20]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import matplotlib.patches as mpatches
import seaborn as sns
from matplotlib.colors import TwoSlopeNorm

# Aesthetics
plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "#f9f9f9",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.family": "sans-serif",
    "axes.titlesize": 14,
    "axes.labelsize": 12,
})
MALE_COLOR   = "#4C72B0"
FEMALE_COLOR = "#DD8452"
NEUTRAL_COLOR = "#cccccc"

## CELL 2: Load Data

In [21]:
df = pd.read_csv("biasbios_initial/bias_in_bios.csv")

# Tidy profession names
df["profession_name"] = df["profession_name"].str.replace("_", " ").str.title()

print(df.shape)
df.head()

(396189, 6)


,split,profession,profession_name,gender,gender_name,hard_text
0,train,21,Professor,0,male,He is also the project lead of and major contr...
1,train,13,Nurse,1,female,"She is able to assess, diagnose and treat mino..."
2,train,2,Attorney,1,female,"Prior to law school, Brittni graduated magna c..."
3,train,11,Journalist,0,male,He regularly contributes to India’s First Onli...
4,train,21,Professor,0,male,He completed his medical degree at Northwester...


## CELL 3: Dataset Overview — Total Counts per Profession

In [22]:
fig, ax = plt.subplots(figsize=(12, 6))

order = (df.groupby("profession_name")["profession_name"]
           .count()
           .sort_values(ascending=False)
           .index)

counts = df["profession_name"].value_counts().reindex(order)
bars = ax.bar(order, counts, color=NEUTRAL_COLOR, edgecolor="white", linewidth=0.8)

ax.set_title("Total Biographies per Profession", fontweight="bold", pad=14)
ax.set_ylabel("Count")
ax.set_xlabel("")
ax.set_xticks(range(len(order)))
ax.set_xticklabels(order, rotation=45, ha="right")
ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f"{int(x):,}"))

# Annotate bars
for bar, val in zip(bars, counts):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 300,
            f"{val:,}", ha="center", va="bottom", fontsize=8, color="#444")

plt.tight_layout()
plt.savefig("biasbios_metrics/01_profession_counts.png", dpi=150)
plt.show()

## CELL 4: Gender Breakdown per Profession (Stacked Bar)

In [23]:
gender_counts = (df.groupby(["profession_name", "gender_name"])
                   .size()
                   .unstack(fill_value=0)
                   .reindex(columns=["male", "female"]))

gender_counts = gender_counts.loc[
    gender_counts.sum(axis=1).sort_values(ascending=False).index
]

fig, ax = plt.subplots(figsize=(13, 6))
bottom = np.zeros(len(gender_counts))

for gender, color, label in [("male", MALE_COLOR, "Male"), ("female", FEMALE_COLOR, "Female")]:
    vals = gender_counts[gender].values
    ax.bar(gender_counts.index, vals, bottom=bottom,
           color=color, label=label, edgecolor="white", linewidth=0.5)
    bottom += vals

ax.set_title("Gender Breakdown by Profession (Absolute Counts)", fontweight="bold", pad=14)
ax.set_ylabel("Number of Biographies")
ax.set_xticks(range(len(gender_counts)))
ax.set_xticklabels(gender_counts.index, rotation=45, ha="right")
ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f"{int(x):,}"))
ax.legend(frameon=False)

plt.tight_layout()
plt.savefig("biasbios_metrics/02_gender_breakdown_stacked.png", dpi=150)
plt.show()

## CELL 5: % Male vs % Female (100 % Stacked)

In [24]:
gender_pct = gender_counts.div(gender_counts.sum(axis=1), axis=0) * 100

# Sort by % female for easy reading
gender_pct = gender_pct.sort_values("female", ascending=True)

fig, ax = plt.subplots(figsize=(7, 9))

y = np.arange(len(gender_pct))
ax.barh(y, gender_pct["male"],  color=MALE_COLOR,   label="Male",   edgecolor="white")
ax.barh(y, gender_pct["female"], left=gender_pct["male"],
        color=FEMALE_COLOR, label="Female", edgecolor="white")

# 50 % reference line
ax.axvline(50, color="white", linewidth=1.5, linestyle="--", alpha=0.9)

ax.set_yticks(y)
ax.set_yticklabels(gender_pct.index)
ax.set_xlim(0, 100)
ax.set_xlabel("Percentage")
ax.set_title("Gender Split per Profession (%)", fontweight="bold", pad=14)
ax.xaxis.set_major_formatter(mtick.PercentFormatter())
ax.legend(frameon=False, loc="lower right")

# Annotate % female on right edge
for i, (prof, row) in enumerate(gender_pct.iterrows()):
    ax.text(101, i, f"{row['female']:.0f}% F", va="center", fontsize=8, color=FEMALE_COLOR)

plt.tight_layout()
plt.savefig("biasbios_metrics/03_gender_pct_horizontal.png", dpi=150)
plt.show()

## CELL 6: Probability of Being Male / Female

In [25]:
# P(gender | profession) — clean dot plot
prob_female = gender_pct["female"].sort_values()

fig, ax = plt.subplots(figsize=(8, 7))

colors = [FEMALE_COLOR if v >= 50 else MALE_COLOR for v in prob_female]
ax.hlines(range(len(prob_female)), 0, 100, color="#e0e0e0", linewidth=1)
ax.scatter(prob_female, range(len(prob_female)), color=colors, s=120, zorder=3)
ax.axvline(50, color="#888", linewidth=1.2, linestyle="--")

ax.set_yticks(range(len(prob_female)))
ax.set_yticklabels(prob_female.index)
ax.set_xlim(0, 100)
ax.set_xlabel("P(female | profession)  →")
ax.set_title("Probability of Being Female per Profession", fontweight="bold", pad=14)
ax.xaxis.set_major_formatter(mtick.PercentFormatter())

male_patch   = mpatches.Patch(color=MALE_COLOR,   label="Male-dominated  (< 50% F)")
female_patch = mpatches.Patch(color=FEMALE_COLOR, label="Female-dominated (≥ 50% F)")
ax.legend(handles=[male_patch, female_patch], frameon=False)

plt.tight_layout()
plt.savefig("biasbios_metrics/04_prob_female_dotplot.png", dpi=150)
plt.show()

## CELL 7: Bias Score — Diverging Bar Chart

In [26]:
# Bias score: how far each profession deviates from 50/50
# +100 = all female, -100 = all male, 0 = perfectly balanced
bias = gender_pct["female"] - 50   # range: -50 to +50
bias = bias.sort_values()

colors = [FEMALE_COLOR if v > 0 else MALE_COLOR for v in bias]

fig, ax = plt.subplots(figsize=(9, 8))
bars = ax.barh(range(len(bias)), bias, color=colors, edgecolor="white")
ax.axvline(0, color="#555", linewidth=1.2)

ax.set_yticks(range(len(bias)))
ax.set_yticklabels(bias.index)
ax.set_xlabel("Bias Score  (% female − 50%)   →")
ax.set_title("Gender Bias Score by Profession\n"
             "(negative = over-represented male, positive = over-represented female)",
             fontweight="bold", pad=14)

# Annotate values
for bar, val in zip(bars, bias):
    xpos = val + (0.8 if val >= 0 else -0.8)
    ha = "left" if val >= 0 else "right"
    ax.text(xpos, bar.get_y() + bar.get_height() / 2,
            f"{val:+.1f}%", va="center", ha=ha, fontsize=9)

male_patch   = mpatches.Patch(color=MALE_COLOR,   label="Male-skewed")
female_patch = mpatches.Patch(color=FEMALE_COLOR, label="Female-skewed")
ax.legend(handles=[male_patch, female_patch], frameon=False, loc="lower right")

plt.tight_layout()
plt.savefig("biasbios_metrics/05_bias_diverging.png", dpi=150)
plt.show()

## CELL 8: Absolute Imbalance — "Missing" Representation

In [27]:
# How many extra bios would be needed to reach 50/50?
gender_counts["total"] = gender_counts.sum(axis=1)
gender_counts["expected_each"] = gender_counts["total"] / 2
gender_counts["deficit_female"] = gender_counts["expected_each"] - gender_counts["female"]
# Positive = female under-represented; negative = male under-represented

deficit = gender_counts["deficit_female"].sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(9, 7))
colors = [MALE_COLOR if v > 0 else FEMALE_COLOR for v in deficit]
ax.barh(range(len(deficit)), deficit, color=colors, edgecolor="white")
ax.axvline(0, color="#555", linewidth=1.2)

ax.set_yticks(range(len(deficit)))
ax.set_yticklabels(deficit.index)
ax.set_xlabel("Extra female bios needed  ←  0  →  Extra male bios needed")
ax.xaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f"{abs(int(x)):,}"))
ax.set_title("Absolute Representation Gap\n(bios needed to reach 50/50 balance)",
             fontweight="bold", pad=14)

male_patch   = mpatches.Patch(color=MALE_COLOR,   label="Female under-represented")
female_patch = mpatches.Patch(color=FEMALE_COLOR, label="Male under-represented")
ax.legend(handles=[male_patch, female_patch], frameon=False)

plt.tight_layout()
plt.savefig("biasbios_metrics/06_absolute_gap.png", dpi=150)
plt.show()

## CELL 9: Heatmap — Normalised Counts

In [28]:
# Rows = profession, cols = gender; colour = % of profession total
heatmap_data = gender_pct[["male", "female"]].copy()

fig, ax = plt.subplots(figsize=(4, 9))
sns.heatmap(
    heatmap_data,
    annot=True, fmt=".0f", annot_kws={"size": 10},
    cmap="RdBu",           # red = male-heavy, blue = female-heavy
    center=50,
    vmin=0, vmax=100,
    linewidths=0.5,
    cbar_kws={"label": "% of profession", "shrink": 0.6},
    ax=ax,
)
ax.set_title("Gender Distribution Heatmap\n(% per profession)", fontweight="bold", pad=14)
ax.set_xlabel("")
ax.set_ylabel("")

plt.tight_layout()
plt.savefig("biasbios_metrics/07_gender_heatmap.png", dpi=150)
plt.show()

## CELL 10: Overall Dataset Gender Balance

In [29]:
overall = df["gender_name"].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# Pie
axes[0].pie(overall, labels=overall.index, colors=[MALE_COLOR, FEMALE_COLOR],
            autopct="%1.1f%%", startangle=90,
            wedgeprops={"edgecolor": "white", "linewidth": 2})
axes[0].set_title("Overall Gender Split", fontweight="bold")

# Bar
axes[1].bar(overall.index, overall.values,
            color=[MALE_COLOR, FEMALE_COLOR], edgecolor="white", width=0.5)
for i, (label, val) in enumerate(overall.items()):
    axes[1].text(i, val + 500, f"{val:,}", ha="center", fontsize=11)
axes[1].set_title("Total Biographies by Gender", fontweight="bold")
axes[1].set_ylabel("Count")
axes[1].yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f"{int(x):,}"))
axes[1].spines["top"].set_visible(False)
axes[1].spines["right"].set_visible(False)

plt.tight_layout()
plt.savefig("biasbios_metrics/08_overall_balance.png", dpi=150)
plt.show()

## CELL 11: Summary Table

In [30]:
summary = gender_counts[["male", "female", "total"]].copy()
summary["pct_female"] = (summary["female"] / summary["total"] * 100).round(1)
summary["pct_male"]   = (summary["male" ]   / summary["total" ] * 100).round(1)
summary["bias_score"] = (summary["pct_female"] - 50).round(1)
summary = summary.sort_values("bias_score")

print(summary.to_string())

gender_name         male  female   total  pct_female  pct_male  bias_score
profession_name                                                           
Rapper              1267     136    1403         9.7      90.3       -40.3
Dj                  1275     211    1486        14.2      85.8       -35.8
Surgeon            11573    2014   13587        14.8      85.2       -35.2
Software Engineer   5820    1092    6912        15.8      84.2       -34.2
Composer            4682     917    5599        16.4      83.6       -33.6
Comedian            2215     594    2809        21.1      78.9       -28.9
Architect           7715    2393   10108        23.7      76.3       -26.3
Pastor              1917     606    2523        24.0      76.0       -26.0
Chiropractor        1956     699    2655        26.3      73.7       -23.7
Filmmaker           4691    2305    6996        32.9      67.1       -17.1
Dentist             9437    5148   14585        35.3      64.7       -14.7
Photographer       15603 